# hpc-as-api — Live Deployment Demo

End-to-end test notebook for the Gemma 4 31B deployment on Lakeshore (ga-002).

**What you need:**
- `pip install openai httpx`
- Set `HPC_GATEWAY_URL` and `HPC_GATEWAY_KEY` as environment variables (or paste them in the config cell below)

**Models available:**
| Alias | Model | Node | Context |
|-------|-------|------|---------|
| `gemma4-31b` | google/gemma-4-31B-it | ga-002 (2× A100 80GB, TP2) | 128K |
| `qwen25-vl-72b` | Qwen/Qwen2.5-VL-72B-Instruct-AWQ | ghi2-002 (1× H100) | 64K |

**Sections:**
1. [Batch (non-streaming)](#1-batch)
2. [Streaming](#2-streaming)
3. [Latency benchmark — N sequential requests](#3-latency-benchmark)
4. [Concurrent load test](#4-concurrent-load-test)
5. [Rate limiting](#5-rate-limiting)
6. [Capacity summary](#6-capacity-summary)
7. [OpenAI SDK — drop-in usage](#7-openai-sdk)
8. [Tool Calling (Function Calling)](#8-tool-calling)
9. [Agentic Workflow](#9-agentic-workflow)

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────────
# Set HPC_GATEWAY_URL and HPC_GATEWAY_KEY as environment variables before
# running this notebook.  Never paste a real API key directly into this file.
BASE_URL = os.environ["HPC_GATEWAY_URL"]   # e.g. https://relay.stream.acer.uic.edu:8001/v1
API_KEY  = os.environ["HPC_GATEWAY_KEY"]   # e.g. sk-stream-...
MODEL    = os.getenv("HPC_GATEWAY_MODEL", "gemma4-31b")

print(f"Endpoint : {BASE_URL}")
print(f"Model    : {MODEL}")
print(f"Key      : {API_KEY[:12]}...")

In [ ]:
import httpx, json

# ── Health check ──────────────────────────────────────────────────────────────
health_url = BASE_URL.replace("/v1", "/health")
r = httpx.get(health_url, timeout=10)
print(json.dumps(r.json(), indent=2))

In [ ]:
# ── List available models ─────────────────────────────────────────────────────
r = httpx.get(BASE_URL + "/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=10)
for m in r.json()["data"]:
    print(f"  {m['id']}  (gateway alias: {m.get('gateway_name', m['id'])})")

---
## 1. Batch (non-streaming)

Measures total round-trip latency. Use this when you want the full answer at once.

In [ ]:
import time, httpx, json

def batch(prompt, max_tokens=256, thinking=False, tools=None, tool_choice=None, messages=None):
    """
    Blocking chat completion.  Returns a dict with:
      elapsed, content, reasoning, tool_calls, tokens, tok_per_s
    Pass `messages` directly to send a full conversation (overrides `prompt`).
    Pass `tools` / `tool_choice` to enable function calling.
    """
    if messages is None:
        messages = [{"role": "user", "content": prompt}]

    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "stream": False,
    }
    if thinking:
        payload["chat_template_kwargs"] = {"enable_thinking": True}
    if tools:
        payload["tools"] = tools
    if tool_choice is not None:
        payload["tool_choice"] = tool_choice

    t0 = time.perf_counter()
    r = httpx.post(
        BASE_URL + "/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    elapsed = time.perf_counter() - t0
    data  = r.json()
    msg   = data["choices"][0]["message"]
    usage = data.get("usage", {})
    toks  = usage.get("completion_tokens", 0)
    return {
        "elapsed":    elapsed,
        "content":    msg.get("content") or "",
        "reasoning":  msg.get("reasoning") or "",
        "tool_calls": msg.get("tool_calls") or [],
        "tokens":     toks,
        "tok_per_s":  toks / elapsed if elapsed > 0 else 0,
    }

# Quick sanity check
r = batch("What is the capital of France?", max_tokens=32)
print(f"Answer  : {r['content']}")
print(f"Latency : {r['elapsed']:.2f}s  |  {r['tokens']} tokens  |  {r['tok_per_s']:.1f} tok/s")

In [ ]:
# ── Batch WITH thinking (reasoning mode) ─────────────────────────────────────
MATH_PROMPT = (
    "A train travels 60 km in 40 minutes, then 90 km in 50 minutes. "
    "What is its average speed in km/h? Show your work."
)

r = batch(MATH_PROMPT, max_tokens=512, thinking=True)
print(f"=== Answer ({r['elapsed']:.2f}s, {r['tokens']} tokens, {r['tok_per_s']:.1f} tok/s) ===")
print(r["content"][:600] if r["content"] else "(no content — model still in thinking phase at max_tokens)")
print(f"\n=== Reasoning ({len(r['reasoning'])} chars) ===")
print(r["reasoning"][:600] if r["reasoning"] else "(none — thinking not triggered for this prompt)")

---
## 2. Streaming

Measures TTFT (time to first token) — this is what determines how interactive the model feels.  
Under 1 second feels real-time; 2–3 seconds is acceptable; beyond that users notice the wait.

In [ ]:
import time, json, httpx

def stream(prompt, max_tokens=256, thinking=False, print_tokens=True,
           tools=None, tool_choice=None, messages=None):
    """
    Stream a chat completion.  Returns a dict with:
      ttft, total, content, reasoning, tool_calls, chunks, tok_per_s
    Pass `messages` directly to send a full conversation (overrides `prompt`).
    Pass `tools` / `tool_choice` to enable function calling.
    """
    if messages is None:
        messages = [{"role": "user", "content": prompt}]

    payload = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "stream": True,
    }
    if thinking:
        payload["chat_template_kwargs"] = {"enable_thinking": True}
    if tools:
        payload["tools"] = tools
    if tool_choice is not None:
        payload["tool_choice"] = tool_choice

    t0 = time.perf_counter()
    ttft = None
    content = ""
    reasoning = ""
    # Accumulate streaming tool_calls deltas into a list of call objects
    tool_calls_acc: dict[int, dict] = {}
    n_chunks = 0
    finish_reason = None

    with httpx.Client(timeout=120) as client:
        with client.stream(
            "POST",
            BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json=payload,
        ) as resp:
            for line in resp.iter_lines():
                if not line.startswith("data:"):
                    continue
                d = line[5:].strip()
                if d == "[DONE]":
                    break
                try:
                    chunk = json.loads(d)
                except json.JSONDecodeError:
                    continue
                choice = chunk["choices"][0]
                delta = choice.get("delta", {})
                if choice.get("finish_reason"):
                    finish_reason = choice["finish_reason"]

                tok  = delta.get("content", "") or ""
                rtok = delta.get("reasoning_content", "") or ""
                tc_deltas = delta.get("tool_calls") or []

                if tok or rtok or tc_deltas:
                    if ttft is None:
                        ttft = time.perf_counter() - t0
                    n_chunks += 1

                content   += tok
                reasoning += rtok
                if print_tokens and tok:
                    print(tok, end="", flush=True)

                # Assemble tool_calls from streaming deltas
                for tc in tc_deltas:
                    idx = tc.get("index", 0)
                    if idx not in tool_calls_acc:
                        tool_calls_acc[idx] = {
                            "id": tc.get("id", ""),
                            "type": tc.get("type", "function"),
                            "function": {"name": "", "arguments": ""},
                        }
                    fn = tc.get("function", {})
                    if fn.get("name"):
                        tool_calls_acc[idx]["function"]["name"] += fn["name"]
                    if fn.get("arguments"):
                        tool_calls_acc[idx]["function"]["arguments"] += fn["arguments"]
                    if tc.get("id"):
                        tool_calls_acc[idx]["id"] = tc["id"]

    total = time.perf_counter() - t0
    tool_calls = [tool_calls_acc[i] for i in sorted(tool_calls_acc)]
    return {
        "ttft": ttft,
        "total": total,
        "content": content,
        "reasoning": reasoning,
        "tool_calls": tool_calls,
        "finish_reason": finish_reason,
        "chunks": n_chunks,
        "tok_per_s": n_chunks / total if total > 0 else 0,
    }

print("stream() helper defined.")

In [ ]:
print("=== Response ===")
r = stream("Explain the twin paradox in three sentences.", max_tokens=200)
print(f"\n\nTTFT    : {r['ttft']:.3f}s")
print(f"Total   : {r['total']:.2f}s")
print(f"Chunks  : {r['chunks']}")
print(f"Tok/s   : {r['tok_per_s']:.1f}")

In [ ]:
# reasoning_content tokens are prefixed with [think] so you can see them inline
print("=== Thinking tokens + Response ===")
r = stream(MATH_PROMPT, max_tokens=600, thinking=True)
print(f"\n\nTTFT       : {r['ttft']:.3f}s")
print(f"Total      : {r['total']:.2f}s")
print(f"Content    : {len(r['content'])} chars")
print(f"Reasoning  : {len(r['reasoning'])} chars")

---
## 3. Latency benchmark — N sequential requests

Measures TTFT distribution over multiple requests.

In [ ]:
import statistics

PROMPTS = [
    "What is 2 + 2?",
    "Name the first five elements of the periodic table.",
    "Summarize the French Revolution in one sentence.",
    "What is the derivative of x^2?",
    "Who wrote Pride and Prejudice?",
]

N_REPS = 2
results = []

for rep in range(N_REPS):
    for p in PROMPTS:
        r = stream(p, max_tokens=64, print_tokens=False)
        results.append(r)
        print(f"  TTFT={r['ttft']:.3f}s  total={r['total']:.2f}s  tok/s={r['tok_per_s']:.1f}  [{p[:40]}]")

ttfts  = [r["ttft"] for r in results]
totals = [r["total"] for r in results]
tps    = [r["tok_per_s"] for r in results]

print(f"\n{'':=<60}")
print(f"Requests : {len(results)}")
print(f"TTFT     : mean={statistics.mean(ttfts):.3f}s  p50={sorted(ttfts)[len(ttfts)//2]:.3f}s  max={max(ttfts):.3f}s")
print(f"Total    : mean={statistics.mean(totals):.2f}s")
print(f"Tok/s    : mean={statistics.mean(tps):.1f}  max={max(tps):.1f}")

---
## 4. Concurrent load test — how many users at ≤1s TTFT?

Fires N requests simultaneously and records each TTFT.  
vLLM uses continuous batching — TTFT grows as the batch fills up.

**Interpretation:**  
`✓` TTFT ≤ 1s (interactive target)  |  `~` 1–3s (acceptable)  |  `✗` > 3s (slow)

In [ ]:
import asyncio, time, json, httpx

CONCURRENT_PROMPT = "Explain the concept of entropy in thermodynamics in 2 sentences."

async def astream_ttft(client, prompt, max_tokens=128):
    payload = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "stream": True}
    t0 = time.perf_counter()
    ttft = None
    n_chunks = 0
    try:
        async with client.stream(
            "POST", BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json=payload,
        ) as resp:
            async for line in resp.aiter_lines():
                if not line.startswith("data:"): continue
                d = line[5:].strip()
                if d == "[DONE]": break
                try: chunk = json.loads(d)
                except: continue
                if chunk["choices"][0].get("delta", {}).get("content"):
                    if ttft is None: ttft = time.perf_counter() - t0
                    n_chunks += 1
    except Exception as e:
        return {"error": str(e), "ttft": None, "total": time.perf_counter() - t0}
    total = time.perf_counter() - t0
    return {"ttft": ttft, "total": total, "chunks": n_chunks,
            "tok_per_s": n_chunks / total if total > 0 else 0}


async def run_concurrent(n, prompt):
    async with httpx.AsyncClient(timeout=120) as client:
        t0 = time.perf_counter()
        results = await asyncio.gather(*[astream_ttft(client, prompt) for _ in range(n)])
        return results, time.perf_counter() - t0


for n_concurrent in [1, 2, 4, 8, 16, 32]:
    results, wall = await run_concurrent(n_concurrent, CONCURRENT_PROMPT)
    ttfts  = [r["ttft"] for r in results if r.get("ttft") is not None]
    errors = sum(1 for r in results if "error" in r)
    if ttfts:
        mean_ttft = statistics.mean(ttfts)
        p95 = sorted(ttfts)[int(len(ttfts) * 0.95)]
        sym = "\u2713" if mean_ttft <= 1.0 else ("~" if mean_ttft <= 3.0 else "\u2717")
        print(f"  n={n_concurrent:3d}  mean_TTFT={mean_ttft:.2f}s  p95={p95:.2f}s  wall={wall:.2f}s  err={errors}  {sym}")
    else:
        print(f"  n={n_concurrent:3d}  ALL ERRORS  wall={wall:.2f}s")

---
## 5. Rate limiting

The proxy enforces a per-caller sliding-window rate limit.  
- Default: `PROXY_RATE_LIMIT_REQUESTS` per window (global)  
- Per-key: `PROXY_RATE_LIMIT_REQUESTS_<NAME>` — e.g. set `PROXY_RATE_LIMIT_REQUESTS_DEMO=20` to cap the demo key at 20 req/min  

Exceeding the limit returns HTTP 429.

In [ ]:
# Fire 25 rapid 1-token requests — shows 429 if the key is rate-limited
import httpx

n_ok = 0
n_429 = 0

with httpx.Client(timeout=10) as client:
    for i in range(25):
        r = client.post(
            BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json={"model": MODEL, "messages": [{"role": "user", "content": "hi"}],
                  "max_tokens": 1, "stream": False},
        )
        if r.status_code == 429:
            n_429 += 1
            print(f"  [{i+1:02d}] 429 — {r.json().get('detail', '')}")
        else:
            n_ok += 1

print(f"\nOK: {n_ok}  |  Rate-limited: {n_429}")

---
## 6. Capacity summary — Gemma 4 31B, 2× A100 SXM4, TP2

In [ ]:
print("""
Measured on 2026-06-15 — Gemma 4 31B, 2× A100 SXM4 80GB, TP2, no quant, BF16
Gateway: relay.stream.acer.uic.edu:8001  →  Globus Compute  →  ga-002:8001

+--------------------------+--------------------------------------------+
| Metric                   | Measured value                             |
+--------------------------+--------------------------------------------+
| Decode throughput        | 25–38 tok/s (single user)                  |
| TTFT (relay, streaming)  | 0.5–1.2s (single user, warm cache)         |
| TTFT (relay, batch)      | 2.3s round-trip (2-token answer)           |
| Thinking mode TTFT       | 0.5s (streaming), 12s total (512 tok)      |
| Reasoning tokens         | Forwarded as reasoning_content in SSE ✓    |
+--------------------------+--------------------------------------------+
| Concurrent capacity      |                                            |
|  Interactive (TTFT ≤1s)  | 1–2 concurrent users                       |
|  Acceptable  (TTFT ≤4s)  | 3–4 concurrent users                       |
|  Slow        (TTFT >4s)  | 8+ concurrent users                        |
+--------------------------+--------------------------------------------+
| vLLM limits              |                                            |
|  --max-num-seqs 128      | Up to 128 requests queued                  |
|  Context window          | 128K tokens                                |
|  KV cache (82 GiB free)  | ~6 full-context (128K) sessions            |
+--------------------------+--------------------------------------------+

Practical guidance:
  - Self-paced homework: students stagger naturally → fast for everyone
  - Synchronized demos: warn class not to submit simultaneously
  - TTFT bottleneck: Globus worker startup + relay round-trip (~0.4s baseline)
  - Throughput bottleneck: GPU compute (38 tok/s single user, shared across all)

Rate limiting defaults (proxy-env):
  PROXY_RATE_LIMIT_REQUESTS=10000  # per-caller per 60s (amplify key)
  PROXY_RATE_LIMIT_REQUESTS_DEMO=20  # demo key: 20 req/min
""")

---
## 7. OpenAI SDK — drop-in usage

Any student can use the official `openai` library — just point it at this gateway.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=32,
)
print("Non-streaming:", resp.choices[0].message.content)

print("\nStreaming: ", end="")
for chunk in client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    max_tokens=32,
    stream=True,
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)
print()

In [ ]:
# Gemma 4 thinking mode — extra_body passes through arbitrary JSON fields
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": MATH_PROMPT}],
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
msg = resp.choices[0].message
print("Content  :", (msg.content or "")[:300])
reasoning = getattr(msg, "reasoning", None) or (msg.model_extra or {}).get("reasoning", "")
print("Reasoning:", (reasoning or "")[:300])

---
## 8. Tool Calling (Function Calling)

hpc-as-api forwards `tools` and `tool_choice` to vLLM unchanged.  Any model that supports
function calling (Gemma 4, Qwen 2.5, etc.) can call tools via the gateway — both batch and streaming.

### How it works
1. You describe tools as JSON schemas in the `tools` array
2. The model decides whether to call a tool and emits `finish_reason: "tool_calls"` with arguments
3. Your code executes the actual function and feeds the result back as a `tool` role message
4. The model generates a final answer using the tool result

This works over the HPC gateway exactly like it does against OpenAI — no special gateway configuration needed.

In [ ]:
import json

# ── Define tools ──────────────────────────────────────────────────────────────
# Two mock scientific tools the model can call.
# In a real HPC workflow these would invoke actual computations.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_simulation_status",
            "description": (
                "Returns the current status of an HPC simulation job, "
                "including elapsed time and convergence residual."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "job_id": {
                        "type": "string",
                        "description": "The SLURM job ID of the simulation",
                    }
                },
                "required": ["job_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Returns the current temperature and conditions for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Chicago'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit. Default: fahrenheit",
                    },
                },
                "required": ["city"],
            },
        },
    },
]

# ── Mock tool implementations ─────────────────────────────────────────────────
def get_simulation_status(job_id: str) -> dict:
    """Fake: returns mock HPC job status."""
    return {
        "job_id": job_id,
        "state": "RUNNING",
        "elapsed_seconds": 1843,
        "residual": 3.7e-5,
        "iterations": 412,
        "convergence": "not yet — target 1e-6",
    }

def get_weather(city: str, unit: str = "fahrenheit") -> dict:
    """Fake: returns mock weather."""
    data = {
        "Chicago": {"temp_f": 72, "temp_c": 22, "conditions": "Partly cloudy"},
        "New York": {"temp_f": 68, "temp_c": 20, "conditions": "Sunny"},
    }
    d = data.get(city, {"temp_f": 65, "temp_c": 18, "conditions": "Unknown"})
    temp = d["temp_f"] if unit == "fahrenheit" else d["temp_c"]
    return {"city": city, "temperature": temp, "unit": unit, "conditions": d["conditions"]}

TOOL_REGISTRY = {
    "get_simulation_status": get_simulation_status,
    "get_weather": get_weather,
}

def dispatch_tool(tool_call: dict) -> str:
    """Execute a tool_call dict returned by the model; return JSON result string."""
    name = tool_call["function"]["name"]
    args = json.loads(tool_call["function"]["arguments"])
    fn   = TOOL_REGISTRY.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    result = fn(**args)
    return json.dumps(result)

print("Tools defined:", [t["function"]["name"] for t in TOOLS])

### 8a. Tool calling — batch (non-streaming)

Simple single-turn: model receives tools, decides to call one, we execute it and send the result back.

In [ ]:
# ── Turn 1: model decides to call a tool ─────────────────────────────────────
messages = [
    {"role": "user", "content": "What's the weather like in Chicago right now?"}
]

r1 = batch(
    prompt=None,
    messages=messages,
    max_tokens=256,
    tools=TOOLS,
    tool_choice="auto",
)

print(f"finish_reason : {r1.get('finish_reason', 'N/A')}  (expected: tool_calls)")
print(f"tool_calls    : {json.dumps(r1['tool_calls'], indent=2)}")

if not r1["tool_calls"]:
    print("\n⚠  Model did not call a tool — it may have answered directly:")
    print(r1["content"])
else:
    # ── Turn 2: execute the tool and send the result back ─────────────────────
    tc = r1["tool_calls"][0]
    tool_result = dispatch_tool(tc)
    print(f"\nTool result: {tool_result}")

    messages = messages + [
        # The assistant's turn that contained the tool_call
        {
            "role": "assistant",
            "content": r1["content"] or None,
            "tool_calls": r1["tool_calls"],
        },
        # The tool result
        {
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": tool_result,
        },
    ]

    r2 = batch(prompt=None, messages=messages, max_tokens=256)
    print(f"\n=== Final answer ({r2['elapsed']:.2f}s) ===")
    print(r2["content"])

### 8b. Tool calling — streaming

Same flow but the assistant turn is streamed token-by-token.  
Tool call arguments arrive as partial JSON deltas; `stream()` reassembles them automatically.

In [ ]:
# ── Turn 1: stream the assistant's tool_call decision ────────────────────────
messages = [
    {"role": "user", "content": "Check the status of HPC job 405639 for me."}
]

print("Streaming turn 1 (tool decision) ...")
r1 = stream(
    prompt=None,
    messages=messages,
    max_tokens=256,
    tools=TOOLS,
    tool_choice="auto",
    print_tokens=True,
)
print()
print(f"finish_reason : {r1['finish_reason']}  (expected: tool_calls)")
print(f"tool_calls    : {json.dumps(r1['tool_calls'], indent=2)}")

if not r1["tool_calls"]:
    print("⚠  Model answered directly (no tool call):", r1["content"])
else:
    tc = r1["tool_calls"][0]
    tool_result = dispatch_tool(tc)
    print(f"Tool result   : {tool_result}")

    messages = messages + [
        {
            "role": "assistant",
            "content": r1["content"] or None,
            "tool_calls": r1["tool_calls"],
        },
        {
            "role": "tool",
            "tool_call_id": tc["id"],
            "content": tool_result,
        },
    ]

    # ── Turn 2: stream the final answer ──────────────────────────────────────
    print("\nStreaming turn 2 (final answer) ...\n")
    r2 = stream(prompt=None, messages=messages, max_tokens=512, print_tokens=True)
    print(f"\n\nTTFT  : {r2['ttft']:.3f}s  |  Total: {r2['total']:.2f}s")

### 8c. Tool calling via the OpenAI SDK

The `openai` library handles all the delta assembly and message construction for you.

In [ ]:
from openai import OpenAI

oai = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Turn 1 — let the model choose whether to call a tool
messages = [
    {"role": "user", "content": "What is the weather in New York in celsius?"}
]

resp1 = oai.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=TOOLS,
    tool_choice="auto",
    max_tokens=256,
)

msg1 = resp1.choices[0].message
print(f"finish_reason : {resp1.choices[0].finish_reason}")
print(f"tool_calls    : {msg1.tool_calls}")

if msg1.tool_calls:
    messages.append(msg1)  # assistant turn (SDK object works directly here)

    for tc in msg1.tool_calls:
        result = dispatch_tool({"function": {"name": tc.function.name,
                                             "arguments": tc.function.arguments}})
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result,
        })
        print(f"Tool {tc.function.name}({tc.function.arguments}) → {result}")

    # Turn 2 — final answer
    resp2 = oai.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=256,
    )
    print(f"\n=== Final answer ===\n{resp2.choices[0].message.content}")
else:
    print("Model answered directly:", msg1.content)

---
## 9. Agentic Workflow

An **agent loop** repeatedly calls the model until it stops issuing tool calls.
This lets the model autonomously plan and execute multi-step tasks — querying tools,
reasoning about the results, and calling more tools as needed.

```
User message
    │
    ▼
[Model] ──tool_calls──▶ [dispatch_tool()] ──result──▶ [Model]
    │                                                      │
    │◀─────────────── loop until finish_reason="stop" ────┘
    ▼
Final answer streamed to user
```

**Example task:** The model must check two things and summarise them — it will
call `get_weather` and `get_simulation_status` in separate turns (or potentially
in parallel if it chooses to), then write a combined report.

In [ ]:
def run_agent(user_message: str, tools: list, max_turns: int = 10,
              max_tokens: int = 512, verbose: bool = True) -> str:
    """
    A simple agent loop over the HPC gateway.

    Runs until the model stops calling tools (finish_reason == "stop")
    or `max_turns` is reached.  The final assistant text is returned.

    Each iteration:
      1. Call the model (streaming)
      2. If tool_calls present → dispatch each → append tool results → loop
      3. If no tool_calls → stream final answer and return
    """
    messages = [{"role": "user", "content": user_message}]
    final_answer = ""

    for turn in range(1, max_turns + 1):
        if verbose:
            print(f"\n{'─'*60}")
            print(f"  Turn {turn} — calling model ...")
            print(f"{'─'*60}")

        r = stream(
            prompt=None,
            messages=messages,
            max_tokens=max_tokens,
            tools=tools,
            tool_choice="auto",
            print_tokens=verbose,
        )

        if verbose:
            print()

        if r["tool_calls"]:
            # Append the assistant's tool_call turn to history
            messages.append({
                "role": "assistant",
                "content": r["content"] or None,
                "tool_calls": r["tool_calls"],
            })

            # Dispatch all tool calls and append results
            for tc in r["tool_calls"]:
                result = dispatch_tool(tc)
                fn_name = tc["function"]["name"]
                fn_args = tc["function"]["arguments"]
                if verbose:
                    print(f"  ► {fn_name}({fn_args})")
                    print(f"    ← {result}")
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": result,
                })

        else:
            # No tool calls — model is done
            final_answer = r["content"]
            if verbose:
                print(f"\n✓ Agent finished in {turn} turn(s).")
                print(f"  TTFT: {r['ttft']:.3f}s  |  Total: {r['total']:.2f}s")
            break

    return final_answer

print("run_agent() defined.")

### 9a. Multi-step agent — weather + HPC job status

The model is asked to gather two pieces of information and write a combined report.
Watch how it calls tools sequentially (or in parallel) and synthesises the results.

In [ ]:
answer = run_agent(
    user_message=(
        "I'm running a CFD simulation on Lakeshore (job ID 405639) and I'm in Chicago. "
        "Please check the current weather here and check the status of my simulation job, "
        "then give me a brief combined status report."
    ),
    tools=TOOLS,
    max_turns=6,
    max_tokens=512,
)

### 9b. Forced tool use — `tool_choice="required"`

Setting `tool_choice="required"` forces the model to always call at least one tool,
even for queries it might otherwise answer from memory.
Useful for workflows that must always go through a verified data source.

In [ ]:
# tool_choice="required" — model MUST call a tool on this turn
r = batch(
    prompt=None,
    messages=[{"role": "user", "content": "What is the weather in Chicago?"}],
    max_tokens=256,
    tools=TOOLS,
    tool_choice="required",   # never skip tools
)
print(f"finish_reason : {r.get('finish_reason')}")
print(f"tool_calls    : {json.dumps(r['tool_calls'], indent=2)}")

### 9c. Specific tool selection — `tool_choice={"type": "function", "function": {"name": "..."}}`

You can force the model to call a specific tool by name.
Useful for structured data extraction — guarantee the model always returns the tool's schema.

In [ ]:
# Force a specific tool — model MUST call get_simulation_status
r = batch(
    prompt=None,
    messages=[{"role": "user", "content": "Tell me about my job."}],
    max_tokens=256,
    tools=TOOLS,
    tool_choice={"type": "function", "function": {"name": "get_simulation_status"}},
)
print(f"finish_reason : {r.get('finish_reason')}")
print(f"tool called   : {r['tool_calls'][0]['function']['name'] if r['tool_calls'] else 'none'}")
print(f"arguments     : {r['tool_calls'][0]['function']['arguments'] if r['tool_calls'] else 'n/a'}")

---
## Student Quick-Start

Ask your instructor for the API key and endpoint URL, then:

```bash
export HPC_GATEWAY_URL="https://<ask-instructor>/v1"
export HPC_GATEWAY_KEY="sk-stream-<ask-instructor>"
```

### Basic usage
```python
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["HPC_GATEWAY_URL"],
    api_key=os.environ["HPC_GATEWAY_KEY"],
)

for chunk in client.chat.completions.create(
    model="gemma4-31b",
    messages=[{"role": "user", "content": "Hello! What can you do?"}],
    stream=True,
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)
```

### Thinking / reasoning mode
```python
resp = client.chat.completions.create(
    model="gemma4-31b",
    messages=[{"role": "user", "content": "Solve: x^2 - 5x + 6 = 0"}],
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
print(resp.choices[0].message.content)
```

### Tool calling
```python
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            },
            "required": ["city"],
        },
    },
}]

resp = client.chat.completions.create(
    model="gemma4-31b",
    messages=[{"role": "user", "content": "What's the weather in Chicago?"}],
    tools=tools,
    tool_choice="auto",
    max_tokens=256,
)
msg = resp.choices[0].message
if msg.tool_calls:
    print("Tool call:", msg.tool_calls[0].function.name,
          msg.tool_calls[0].function.arguments)
```